In [ ]:
from pathlib import Path
import numpy as np
import pickle
import torch
import json
from tqdm import tqdm

In [ ]:
# Load embeddings

ckpt_dir = Path("../embs/<run-id!!!! MUST FILL>")  # transductive/inductive depends on what the model was trained on

ckpt_paths = sorted(ckpt_dir.iterdir(), key=lambda x: int(x.stem.split("_")[-1]))
print(f"Found {len(ckpt_paths)} checkpoints")

ids = None
embs = []  # List of embeddings from different checkpoints
for path in tqdm(ckpt_paths):
    emb_data = torch.load(path, weights_only=False, map_location="cpu")

    embs.append(emb_data["embs"].float().numpy())
    if ids is None:
        ids = emb_data["ids"]
    else:
        assert (ids == emb_data["ids"]).all()

In [ ]:
# Load splits (choose zero-shot or not here)

#splits_file = Path("../splits/ibl_transductive.pkl")  # non-zero-shot!!
splits_file = Path("../splits/ibl_inductive.pkl")  # zero-shot!! 
with open(splits_file, "rb") as f:
    splits_data = pickle.load(f)

label_map = splits_data["label_map"]
split = splits_data["splits"]

In [ ]:
# Remove UUIDs not known to splits

def to_mask(check_ids, base_ids):
    """
    Returns a bool array indicating if elements of `check_ids` are 
    present in `base_ids`. 
    Output shape: (len(check_ids),)
    """
    base_ids_set = set(base_ids)
    return np.array([x in base_ids_set for x in check_ids])

# Subset units based on what labels are available
valid_id_mask = to_mask(ids, label_map.index.values)
print(f"Valid UUID stats: {valid_id_mask.mean()=}, {(~valid_id_mask).sum()=}")

valid_ids = ids[valid_id_mask]
#valid_labels = data["curated_cluster_cosmos"][valid_id_mask]
valid_labels = label_map.loc[valid_ids].to_numpy()
valid_embs = []
for emb in tqdm(embs):
    valid_embs.append(emb[valid_id_mask])

In [ ]:
from cuml.linear_model import LogisticRegression
from cuml.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report
from tqdm import tqdm
import numpy as np
import ray

seed = 42

def balanced_resampling(X, y):
    resample = RandomOverSampler(random_state=seed)
    resample_idx, _ = resample.fit_resample(np.arange(len(y)).reshape(-1, 1), y)
    resample_idx = resample_idx.ravel()
    return X[resample_idx], y[resample_idx]

def logreg_scores(X, y, train_mask, val_mask, test_mask):
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    X_train, y_train = balanced_resampling(X_train, y_train)

    scaler = StandardScaler()
    clf = LogisticRegression(
        max_iter=1000,
        tol=1e-5,
        C=1.0,
        verbose=0,
    )
    clf.fit(scaler.fit_transform(X_train), y_train)
    pred_val = clf.predict(scaler.transform(X_val))
    pred_test = clf.predict(scaler.transform(X_test))

    return {
        "val_f1": f1_score(y_val, pred_val, average='macro'),
        "val_bacc": balanced_accuracy_score(y_val, pred_val),
        "val_report": classification_report(y_val, pred_val),
        "test_f1": f1_score(y_test, pred_test, average='macro'),
        "test_bacc": balanced_accuracy_score(y_test, pred_test),
        "test_report": classification_report(y_test, pred_test),
        "test_pred": pred_test,
        "test_true": y_test,
    }

@ray.remote(num_cpus=1, num_gpus=0.1)
def ray_logreg_scores(X, y, train_mask, val_mask, test_mask):
    return logreg_scores(X, y, train_mask, val_mask, test_mask)

ray.init(address="local", num_cpus=32, num_gpus=1, ignore_reinit_error=True)


train_ids = split["train"]
np.random.seed(0)
label_ratio = 1.0
#label_ratio = 0.5
#label_ratio = 0.25
#label_ratio = 0.025
print(f"Label ratio = {label_ratio}")
train_ids = np.random.permutation(train_ids)[: int(label_ratio * len(train_ids))]

train_mask = to_mask(valid_ids, train_ids)
val_mask = to_mask(valid_ids, split["val"])
test_mask = to_mask(valid_ids, split["test"])

y = valid_labels
futures = []
for i, X in tqdm(enumerate(valid_embs), total=len(valid_embs)):
    _future = ray_logreg_scores.remote(X, y, train_mask, val_mask, test_mask)
    futures.append(_future)

scores_list = []
for i, future in enumerate(futures):
    scores = ray.get(future)
    scores["index"] = i
    scores_list.append(scores)
    print(
        f"Index {i} | "
        f"Validation - bacc: {scores['val_bacc']:.4f}, F1: {scores['val_f1']:.4f} | "
        f"Test - bacc: {scores['test_bacc']:.4f}, F1: {scores['test_f1']:.4f}"
    )

best_scores = sorted(scores_list, key=lambda x: x["val_f1"], reverse=True)[0]
print(f"Best index: {best_scores['index']}")
print(f"Validation bal. acc.: {best_scores['val_bacc']:.4f}")
print(f"Validation F1: {best_scores['val_f1']:.4f}")
print(f"Test bal. acc.: {best_scores['test_bacc']:.4f}") 
print(f"Test F1: {best_scores['test_f1']:.4f}")
print(f"Best checkpoint: {ckpt_paths[best_scores['index']].name}")